In [2]:
# ==========================================
# IMPORT LIBRARIES
# ==========================================
import pandas as pd
import numpy as np

from scipy import stats
from scipy.stats import (
    pearsonr, spearmanr,
    ttest_1samp, ttest_ind,
    chi2_contingency, f_oneway,
    norm
)

# ==========================================
# LOAD DATASET
# ==========================================
df = pd.read_excel("Trip_Analysis.xlsx")

print(df.head())
print(df.info())

# ==========================================
# DATA CLEANING
# ==========================================

print("Missing Values")
print(df.isnull().sum())

print("Duplicate Rows:", df.duplicated().sum())

df.drop_duplicates(inplace=True)

# ==========================================
# LINEAR ALGEBRA
# ==========================================

vector_a = df["Trip_Distance"].values[:5]
vector_b = df["Fare_Amount"].values[:5]

print("\nVector A")
print(vector_a)

print("\nVector B")
print(vector_b)

# Dot Product
dot_product = np.dot(vector_a, vector_b)
print("Dot Product =", dot_product)

# Matrix Formation
matrix = df[[
    "Trip_Distance",
    "Fare_Amount",
    "Surge_Multiplier"
]].head(3).values

print("\nMatrix")
print(matrix)

# Determinant
det = np.linalg.det(matrix)
print("Determinant =", det)

# Eigenvalues & Eigenvectors
eigenvalues, eigenvectors = np.linalg.eig(matrix)

print("\nEigenvalues")
print(eigenvalues)

print("\nEigenvectors")
print(eigenvectors)

# ==========================================
# PROBABILITY
# ==========================================

# Basic Probability
prob_high_rating = len(df[df["Customer_Rating"] >= 4]) / len(df)

print("\nP(Rating >=4)")
print(prob_high_rating)

# Conditional Probability
peak = df[df["Ride_Time"] == "Peak"]

conditional_prob = (
    len(peak[peak["Customer_Rating"] >= 4])
    / len(peak)
)

print("P(Rating>=4 | Peak)")
print(conditional_prob)

# Bayes Theorem

P_A = len(df[df["Customer_Rating"] >= 4]) / len(df)

P_B = len(df[df["Ride_Time"] == "Peak"]) / len(df)

P_B_given_A = (
    len(df[
        (df["Customer_Rating"] >= 4)
        & (df["Ride_Time"] == "Peak")
    ])
    /
    len(df[df["Customer_Rating"] >= 4])
)

bayes = (P_B_given_A * P_A) / P_B

print("Bayes Probability =", bayes)

# ==========================================
# DESCRIPTIVE STATISTICS
# ==========================================

numeric_cols = [
    "Trip_Distance",
    "Fare_Amount",
    "Surge_Multiplier",
    "Customer_Rating"
]

for col in numeric_cols:

    print(f"\n----- {col} -----")

    print("Mean:", df[col].mean())
    print("Median:", df[col].median())
    print("Mode:", df[col].mode()[0])
    print("Variance:", df[col].var())
    print("Std Dev:", df[col].std())
    print("Skewness:", df[col].skew())
    print("Kurtosis:", df[col].kurt())

# ==========================================
# OUTLIER DETECTION
# ==========================================

column = "Fare_Amount"

# Percentiles
p25 = np.percentile(df[column], 25)
p75 = np.percentile(df[column], 75)

print("\n25th Percentile =", p25)
print("75th Percentile =", p75)

# IQR
Q1 = df[column].quantile(0.25)
Q3 = df[column].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers_iqr = df[
    (df[column] < lower)
    | (df[column] > upper)
]

print("IQR Outliers =", len(outliers_iqr))

# Z Score
z_scores = np.abs(stats.zscore(df[column]))

outliers_z = df[z_scores > 3]

print("Z-score Outliers =", len(outliers_z))

# ==========================================
# RELATIONSHIPS
# ==========================================

# Pearson
pearson_corr, _ = pearsonr(
    df["Trip_Distance"],
    df["Fare_Amount"]
)

print("\nPearson Correlation =", pearson_corr)

# Spearman
spearman_corr, _ = spearmanr(
    df["Trip_Distance"],
    df["Fare_Amount"]
)

print("Spearman Correlation =", spearman_corr)

# Covariance
cov = np.cov(
    df["Trip_Distance"],
    df["Fare_Amount"]
)

print("Covariance Matrix")
print(cov)

# ==========================================
# CENTRAL LIMIT THEOREM
# ==========================================

sample_means = []

for i in range(1000):

    sample = df["Fare_Amount"].sample(
        n=30,
        replace=True
    )

    sample_means.append(sample.mean())

print("\nCLT Mean =", np.mean(sample_means))
print("CLT Std =", np.std(sample_means))

# ==========================================
# BOOTSTRAPPING
# ==========================================

bootstrap_means = []

for i in range(1000):

    sample = df["Fare_Amount"].sample(
        frac=1,
        replace=True
    )

    bootstrap_means.append(sample.mean())

print("Bootstrap Mean =", np.mean(bootstrap_means))

# ==========================================
# POINT ESTIMATION
# ==========================================

point_estimate = df["Fare_Amount"].mean()

print("\nPoint Estimate =", point_estimate)

# ==========================================
# INTERVAL ESTIMATION
# ==========================================

confidence = 0.95

mean = df["Fare_Amount"].mean()
std = df["Fare_Amount"].std()

n = len(df)

z = norm.ppf(0.975)

margin = z * (std / np.sqrt(n))

lower_ci = mean - margin
upper_ci = mean + margin

print("95% CI =", (lower_ci, upper_ci))

# ==========================================
# HYPOTHESIS TESTING
# ==========================================

# Z TEST
z_stat = (
    mean - 250
) / (std / np.sqrt(n))

print("\nZ Statistic =", z_stat)

# T TEST
t_stat, p_value = ttest_1samp(
    df["Fare_Amount"],
    250
)

print("T-Test")
print(t_stat, p_value)

# ==========================================
# CHI-SQUARE TEST
# ==========================================

contingency = pd.crosstab(
    df["Ride_Time"],
    df["Ride_Category"]
)

chi2, p, dof, expected = chi2_contingency(
    contingency
)

print("\nChi-Square")
print("Chi2 =", chi2)
print("p-value =", p)

# ==========================================
# F TEST
# ==========================================

peak_fare = df[
    df["Ride_Time"] == "Peak"
]["Fare_Amount"]

nonpeak_fare = df[
    df["Ride_Time"] == "Non-Peak"
]["Fare_Amount"]

f_stat = np.var(
    peak_fare,
    ddof=1
) / np.var(
    nonpeak_fare,
    ddof=1
)

print("\nF Statistic =", f_stat)

# ==========================================
# ANOVA
# ==========================================

groups = []

for category in df["Ride_Category"].unique():

    groups.append(
        df[
            df["Ride_Category"] == category
        ]["Fare_Amount"]
    )

anova_result = f_oneway(*groups)

print("\nANOVA")
print(anova_result)

# ==========================================
# A/B TESTING
# ==========================================

A = df[
    df["Ride_Time"] == "Peak"
]["Fare_Amount"]

B = df[
    df["Ride_Time"] == "Non-Peak"
]["Fare_Amount"]

ab_test = ttest_ind(
    A,
    B,
    equal_var=False
)

print("\nA/B Testing")
print(ab_test)

   Trip_Distance  Fare_Amount Ride_Category  Surge_Multiplier  \
0              8   270.569350        Shared               2.5   
1             21   275.161032       Economy               2.5   
2             16   359.748971       Economy               2.0   
3             12   264.044264       Premium               1.5   
4              9   225.256916        Shared               2.0   

   Customer_Rating Ride_Time  
0                3  Non-Peak  
1                3  Non-Peak  
2                5  Non-Peak  
3                5  Non-Peak  
4                2  Non-Peak  
<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Trip_Distance     200 non-null    int64  
 1   Fare_Amount       200 non-null    float64
 2   Ride_Category     200 non-null    str    
 3   Surge_Multiplier  200 non-null    float64
 4   Customer_Rating   200 non-null    int64  
 